In [ ]:
!pip install torch transformers datasets scikit-learn tqdm

In [ ]:
"""
=============================================================================
SEIR-DNet: Sentence-level Embedding Invariance and Robustness Defense Network
=============================================================================

Paper concept: A novel transformer-based defense against sentence-level
adversarial attacks (SCPN, SEA, paraphrasing, back-translation rewrites).

Key components:
  1. Shared BERT encoder   – processes all N sentence variants in parallel
  2. SAM                   – Semantic Alignment Module (cross-attention over variants)
  3. LSCR                  – Latent Semantic Consistency Regularizer (novel loss)
  4. HRL                   – Hierarchical Representation Layer (word→phrase→sentence)
  5. PVN                   – Prediction Variance Normalizer (novel component)
  6. Combined loss         – L_cls + λ₁·L_consist + λ₂·L_sim + λ₃·L_struct

Tested on: SST-2 (Stanford Sentiment Treebank, binary classification)
Requirements: pip install torch transformers datasets scikit-learn tqdm

Author: SEIR-DNet reference implementation
=============================================================================
"""

# ──────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTS
# ──────────────────────────────────────────────────────────────────────────────
import os, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup
from datasets import load_dataset
from sklearn.metrics import accuracy_score
from tqdm import tqdm

# reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ──────────────────────────────────────────────────────────────────────────────
# 1.  SIMPLE PARAPHRASE / AUGMENTATION HELPERS
#     (In production use Pegasus or back-translation; here we use rule-based
#      lightweight augmentations so the code runs without extra downloads.)
# ──────────────────────────────────────────────────────────────────────────────

SYNONYM_MAP = {
    "good": "great", "great": "excellent", "bad": "terrible",
    "terrible": "awful", "awful": "dreadful", "movie": "film",
    "film": "movie", "story": "narrative", "plot": "storyline",
    "actor": "performer", "beautiful": "stunning", "boring": "dull",
    "fun": "enjoyable", "sad": "melancholic", "happy": "joyful",
    "poor": "weak", "strong": "powerful", "fast": "swift",
    "slow": "sluggish", "amazing": "remarkable", "nice": "pleasant",
}

def synonym_swap(text: str, p: float = 0.25) -> str:
    """Replace up to p-fraction of tokens with synonyms."""
    tokens = text.split()
    out = []
    for t in tokens:
        low = t.lower().strip(".,!?;:'\"")
        if low in SYNONYM_MAP and random.random() < p:
            out.append(SYNONYM_MAP[low])
        else:
            out.append(t)
    return " ".join(out)

def random_deletion(text: str, p: float = 0.10) -> str:
    """Randomly delete words with probability p."""
    tokens = text.split()
    if len(tokens) == 1:
        return text
    return " ".join([t for t in tokens if random.random() > p]) or tokens[0]

def word_shuffle(text: str, window: int = 3) -> str:
    """Locally shuffle words within windows of size `window`."""
    tokens = text.split()
    for i in range(0, len(tokens) - window + 1, window):
        chunk = tokens[i:i + window]
        random.shuffle(chunk)
        tokens[i:i + window] = chunk
    return " ".join(tokens)

def passive_voice_simple(text: str) -> str:
    """
    Approximate syntactic transform: move last noun-like word to front.
    This is a toy stand-in for SCPN; replace with a real parser in production.
    """
    tokens = text.split()
    if len(tokens) >= 4:
        tokens = [tokens[-1]] + tokens[:-1]
    return " ".join(tokens)

def augment_sentence(text: str) -> list:
    """
    Returns [original, paraphrase1, paraphrase2] for a given input.
    In production: replace with Pegasus paraphrase + en→de→en back-translation.
    """
    return [
        text,                              # original
        synonym_swap(text, p=0.3),         # lexical paraphrase
        word_shuffle(random_deletion(text)),# structural paraphrase
    ]


# ──────────────────────────────────────────────────────────────────────────────
# 2.  DATASET
# ──────────────────────────────────────────────────────────────────────────────

class SST2Dataset(Dataset):
    """
    Wraps the HuggingFace SST-2 dataset.
    Each item returns N_VARIANTS tokenised views of the same sentence + label.
    """
    def __init__(self, split: str, tokenizer: BertTokenizer,
                 max_len: int = 128, n_variants: int = 3):
        self.tokenizer  = tokenizer
        self.max_len    = max_len
        self.n_variants = n_variants

        raw = load_dataset("glue", "sst2", split=split)
        self.texts  = [ex["sentence"] for ex in raw]
        self.labels = [ex["label"]    for ex in raw]

    def __len__(self):
        return len(self.texts)

    def _encode(self, text: str):
        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt",
        )
        return enc["input_ids"].squeeze(0), enc["attention_mask"].squeeze(0)

    def __getitem__(self, idx):
        variants = augment_sentence(self.texts[idx])[:self.n_variants]
        # ensure we always have exactly n_variants
        while len(variants) < self.n_variants:
            variants.append(variants[0])

        all_ids, all_masks = [], []
        for v in variants:
            ids, mask = self._encode(v)
            all_ids.append(ids)
            all_masks.append(mask)

        return (
            torch.stack(all_ids),   # (N, L)
            torch.stack(all_masks), # (N, L)
            torch.tensor(self.labels[idx], dtype=torch.long),
        )


# ──────────────────────────────────────────────────────────────────────────────
# 3.  MODEL COMPONENTS
# ──────────────────────────────────────────────────────────────────────────────

# ── 3a. Semantic Alignment Module (SAM) ──────────────────────────────────────
class SemanticAlignmentModule(nn.Module):
    """
    SAM: Given N sentence embeddings (from [CLS] tokens), compute a
    cross-variant attention to align representations.

    Unlike standard contrastive learning, SAM is a *deterministic*
    differentiable module that explicitly pulls all variant embeddings
    toward a single consensus representation via cross-attention pooling.

    Mathematical form:
      A = softmax( Q·Kᵀ / √d ) · V
      where Q = W_q·z_i,  K = V = stack of all zⱼ (j ∈ 1..N)

    This is NOT just contrastive loss — it is an architectural component
    that transforms each embedding conditioned on all others.
    """
    def __init__(self, hidden_size: int, num_heads: int = 4):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_heads,
            batch_first=True,
        )
        self.norm = nn.LayerNorm(hidden_size)
        self.proj = nn.Linear(hidden_size, hidden_size)

    def forward(self, embeddings: torch.Tensor) -> torch.Tensor:
        """
        embeddings: (B, N, D)  — B=batch, N=variants, D=hidden
        returns:    (B, N, D)  — aligned embeddings
        """
        # cross-variant attention: each variant attends to all variants
        aligned, _ = self.attn(embeddings, embeddings, embeddings)
        aligned = self.norm(embeddings + aligned)          # residual
        return self.proj(aligned)


# ── 3b. Hierarchical Representation Layer (HRL) ──────────────────────────────
class HierarchicalRepresentationLayer(nn.Module):
    """
    HRL: Fuses word-level, phrase-level, and sentence-level signals
    from BERT's hidden states.

    BERT outputs hidden states for every token.  HRL extracts:
      - word level    : mean of all token hidden states
      - phrase level  : convolution over local windows of size 3
      - sentence level: [CLS] token representation

    These three signals are fused with learned weights.

    This goes beyond simple mean-pooling or CLS-only approaches.
    """
    def __init__(self, hidden_size: int):
        super().__init__()
        # phrase-level: 1D conv along the sequence dimension
        self.phrase_conv = nn.Conv1d(
            in_channels=hidden_size,
            out_channels=hidden_size,
            kernel_size=3,
            padding=1,
        )
        self.phrase_norm = nn.LayerNorm(hidden_size)

        # fusion: combine [word, phrase, sentence] → single vector
        self.fusion = nn.Sequential(
            nn.Linear(hidden_size * 3, hidden_size * 2),
            nn.GELU(),
            nn.Linear(hidden_size * 2, hidden_size),
            nn.LayerNorm(hidden_size),
        )

    def forward(self, hidden_states: torch.Tensor,
                attention_mask: torch.Tensor,
                cls_embedding: torch.Tensor) -> torch.Tensor:
        """
        hidden_states  : (B, L, D) — all token hidden states from BERT
        attention_mask : (B, L)    — 1 for real tokens, 0 for padding
        cls_embedding  : (B, D)    — [CLS] representation (sentence level)
        returns        : (B, D)    — fused hierarchical representation
        """
        # word-level: masked mean pooling
        mask = attention_mask.unsqueeze(-1).float()          # (B,L,1)
        word_rep = (hidden_states * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

        # phrase-level: conv along sequence then masked mean
        h_t = hidden_states.transpose(1, 2)                  # (B,D,L)
        phrase = self.phrase_conv(h_t).transpose(1, 2)        # (B,L,D)
        phrase = self.phrase_norm(phrase)
        phrase_rep = (phrase * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

        # fusion
        combined = torch.cat([word_rep, phrase_rep, cls_embedding], dim=-1)
        return self.fusion(combined)


# ── 3c. Prediction Variance Normalizer (PVN) ─────────────────────────────────
class PredictionVarianceNormalizer(nn.Module):
    """
    PVN: A novel component that explicitly regularizes the model to produce
    consistent output distributions across all N variant inputs.

    It computes the variance of softmax distributions across N variants and
    penalizes high variance as part of the loss (L_consist term).

    Unlike R-Drop (which applies dropout stochastically), PVN operates
    deterministically on the N semantically-equivalent variants.
    Unlike standard adversarial training, PVN does not require gradient-based
    attack generation at training time.

    L_consist = (1/N) Σᵢ KL( p₀ || pᵢ ) + KL( pᵢ || p₀ )
    where p₀ = prediction on the original sentence.
    """
    def __init__(self, num_labels: int, temperature: float = 2.0):
        super().__init__()
        self.T = temperature
        self.num_labels = num_labels

    def forward(self, logits_list: list) -> torch.Tensor:
        """
        logits_list: list of (B, num_labels) tensors, one per variant
        returns: scalar consistency loss
        """
        if len(logits_list) < 2:
            return torch.tensor(0.0, device=logits_list[0].device)

        # soft probabilities with temperature scaling
        probs = [F.softmax(l / self.T, dim=-1) for l in logits_list]
        anchor = probs[0]   # original sentence is the anchor
        loss = 0.0
        for p in probs[1:]:
            # symmetric KL divergence
            kl_fwd = F.kl_div(
                torch.log(p + 1e-9), anchor, reduction="batchmean"
            )
            kl_rev = F.kl_div(
                torch.log(anchor + 1e-9), p, reduction="batchmean"
            )
            loss = loss + (kl_fwd + kl_rev) / 2.0
        return loss / (len(logits_list) - 1)


# ── 3d. Latent Semantic Consistency Regularizer (LSCR) ───────────────────────
class LatentSemanticConsistencyRegularizer(nn.Module):
    """
    LSCR: The core novel contribution.

    Enforces that semantically equivalent sentences have geometrically
    close representations in the latent space.

    Unlike contrastive learning (which uses positive/negative pairs and
    needs careful negative mining), LSCR:
      1. Uses a learnable projection head W: D → D_proj
      2. Projects all N embeddings into a lower-dimensional semantic space
      3. Minimises both MSE distance and cosine dissimilarity to the anchor

    L_sim = (1/N) Σᵢ [ MSE(ẑ₀, ẑᵢ) + (1 - cos(ẑ₀, ẑᵢ)) ]
    where ẑ = W · z  (projected embedding)

    L_struct = variance of ‖ẑᵢ‖₂ norms across variants
    (encourages norm-consistency, not just direction-consistency)
    """
    def __init__(self, hidden_size: int, proj_dim: int = 256):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(hidden_size, proj_dim),
            nn.GELU(),
            nn.Linear(proj_dim, proj_dim),
            nn.LayerNorm(proj_dim),
        )

    def forward(self, embeddings: torch.Tensor):
        """
        embeddings : (B, N, D)
        returns    : (L_sim, L_struct) both scalars
        """
        B, N, D = embeddings.shape
        proj = self.proj(embeddings.view(B * N, D)).view(B, N, -1)  # (B,N,P)

        anchor = proj[:, 0, :]   # (B, P) — original sentence projection

        l_sim = 0.0
        norms = []
        for i in range(1, N):
            zi = proj[:, i, :]
            mse  = F.mse_loss(zi, anchor.detach())
            cos  = 1.0 - F.cosine_similarity(zi, anchor.detach(), dim=-1).mean()
            l_sim = l_sim + mse + cos
            norms.append(zi.norm(dim=-1))

        l_sim = l_sim / max(N - 1, 1)

        # structural invariance: penalise norm spread across variants
        all_norms = torch.stack(
            [anchor.norm(dim=-1)] + norms, dim=1
        )  # (B, N)
        l_struct = all_norms.var(dim=1).mean()

        return l_sim, l_struct


# ── 3e.  Full SEIR-DNet Model ─────────────────────────────────────────────────
class SEIRDNet(nn.Module):
    """
    Full SEIR-DNet model.

    Diagram of forward pass:
      N variants → BERT (shared) → [CLS] + hidden states
                 → HRL (word/phrase/sentence fusion)
                 → SAM (cross-variant alignment)
                 → LSCR (consistency regularization)
                 → PVN (prediction variance normalization)
                 → Classifier
    """
    def __init__(
        self,
        bert_name:    str   = "bert-base-uncased",
        num_labels:   int   = 2,
        n_variants:   int   = 3,
        proj_dim:     int   = 256,
        sam_heads:    int   = 4,
        pvn_temp:     float = 2.0,
        dropout:      float = 0.1,
    ):
        super().__init__()
        self.n_variants = n_variants

        # Shared BERT encoder
        self.bert = BertModel.from_pretrained(bert_name)
        H = self.bert.config.hidden_size   # 768 for bert-base

        # Novel components
        self.hrl  = HierarchicalRepresentationLayer(H)
        self.sam  = SemanticAlignmentModule(H, num_heads=sam_heads)
        self.lscr = LatentSemanticConsistencyRegularizer(H, proj_dim)
        self.pvn  = PredictionVarianceNormalizer(num_labels, pvn_temp)

        # Classification head
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(H, H // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(H // 2, num_labels),
        )

    def encode_variant(self, input_ids, attention_mask):
        """Encode a single variant: returns (cls_emb, all_hidden, mask)."""
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask,
                        output_hidden_states=False)
        cls_emb     = out.last_hidden_state[:, 0, :]   # [CLS] token
        all_hidden  = out.last_hidden_state              # all tokens
        return cls_emb, all_hidden

    def forward(self, input_ids, attention_mask):
        """
        input_ids      : (B, N, L)
        attention_mask : (B, N, L)
        returns : dict with logits, loss components
        """
        B, N, L = input_ids.shape

        # ── encode all variants through shared BERT + HRL ──
        cls_embs, hrl_embs = [], []
        for i in range(N):
            ids_i  = input_ids[:, i, :]          # (B, L)
            mask_i = attention_mask[:, i, :]     # (B, L)
            cls_i, hidden_i = self.encode_variant(ids_i, mask_i)

            # HRL: fuse word + phrase + sentence level
            hrl_i = self.hrl(hidden_i, mask_i, cls_i)   # (B, D)
            cls_embs.append(cls_i)
            hrl_embs.append(hrl_i)

        # stack to (B, N, D)
        hrl_stack = torch.stack(hrl_embs, dim=1)   # (B, N, D)

        # ── SAM: cross-variant semantic alignment ──
        aligned = self.sam(hrl_stack)              # (B, N, D)

        # ── LSCR: compute consistency and structural losses ──
        l_sim, l_struct = self.lscr(aligned)

        # ── classify each variant ──
        logits_list = []
        for i in range(N):
            logits_i = self.classifier(aligned[:, i, :])
            logits_list.append(logits_i)

        # ── PVN: prediction variance loss ──
        l_consist = self.pvn(logits_list)

        # primary logits = original sentence (index 0)
        primary_logits = logits_list[0]

        return {
            "logits":    primary_logits,
            "all_logits": logits_list,
            "l_sim":     l_sim,
            "l_struct":  l_struct,
            "l_consist": l_consist,
        }


# ──────────────────────────────────────────────────────────────────────────────
# 4.  LOSS FUNCTION
# ──────────────────────────────────────────────────────────────────────────────

class SEIRLoss(nn.Module):
    """
    Combined loss:
      L_total = L_cls + λ₁·L_consist + λ₂·L_sim + λ₃·L_struct

    L_cls     : cross-entropy on primary (original) logits
    L_consist : symmetric KL across N variant predictions (PVN)
    L_sim     : MSE + cosine loss in projected latent space (LSCR)
    L_struct  : norm-variance regularizer (LSCR)
    """
    def __init__(self,
                 lambda_consist: float = 0.5,
                 lambda_sim:     float = 0.3,
                 lambda_struct:  float = 0.1):
        super().__init__()
        self.lam1 = lambda_consist
        self.lam2 = lambda_sim
        self.lam3 = lambda_struct
        self.ce   = nn.CrossEntropyLoss()

    def forward(self, outputs: dict, labels: torch.Tensor):
        l_cls     = self.ce(outputs["logits"], labels)
        l_consist = outputs["l_consist"]
        l_sim     = outputs["l_sim"]
        l_struct  = outputs["l_struct"]

        total = (
            l_cls
            + self.lam1 * l_consist
            + self.lam2 * l_sim
            + self.lam3 * l_struct
        )
        return total, {
            "cls":     l_cls.item(),
            "consist": l_consist.item() if torch.is_tensor(l_consist) else l_consist,
            "sim":     l_sim.item(),
            "struct":  l_struct.item(),
        }


# ──────────────────────────────────────────────────────────────────────────────
# 5.  TRAINING LOOP
# ──────────────────────────────────────────────────────────────────────────────

def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0
    preds_all, labels_all = [], []
    loss_parts = {"cls": 0, "consist": 0, "sim": 0, "struct": 0}

    for ids, masks, labels in tqdm(loader, desc="Train", leave=False):
        ids    = ids.to(device)
        masks  = masks.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(ids, masks)
        loss, parts = criterion(outputs, labels)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        for k in loss_parts:
            loss_parts[k] += parts[k]

        preds = outputs["logits"].argmax(-1).cpu().tolist()
        preds_all.extend(preds)
        labels_all.extend(labels.cpu().tolist())

    n = len(loader)
    return {
        "loss":    total_loss / n,
        "acc":     accuracy_score(labels_all, preds_all),
        **{k: v / n for k, v in loss_parts.items()},
    }


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    preds_all, labels_all = [], []

    for ids, masks, labels in tqdm(loader, desc="Eval ", leave=False):
        ids    = ids.to(device)
        masks  = masks.to(device)
        labels = labels.to(device)

        outputs = model(ids, masks)
        loss, _ = criterion(outputs, labels)
        total_loss += loss.item()

        preds = outputs["logits"].argmax(-1).cpu().tolist()
        preds_all.extend(preds)
        labels_all.extend(labels.cpu().tolist())

    return {
        "loss": total_loss / len(loader),
        "acc":  accuracy_score(labels_all, preds_all),
    }


# ──────────────────────────────────────────────────────────────────────────────
# 6.  ADVERSARIAL ROBUSTNESS EVALUATION
#     Simulates SCPN / SEA-style sentence-level attacks at inference time.
# ──────────────────────────────────────────────────────────────────────────────

def simulate_scpn_attack(text: str) -> str:
    """
    Lightweight simulation of SCPN (Syntactically Controlled Paraphrase Network)
    attacks: reorder words + heavy synonym swap + deletion.
    In production, replace with actual SCPN or SEA attack generation.
    """
    t = word_shuffle(text, window=4)
    t = synonym_swap(t, p=0.5)
    t = random_deletion(t, p=0.15)
    return t


@torch.no_grad()
def evaluate_robustness(model, loader, tokenizer, criterion, device,
                        max_len: int = 128):
    """
    Evaluates model accuracy on SCPN-simulated attacks.
    Reports: clean accuracy, attacked accuracy, attack success rate.
    """
    model.eval()
    clean_preds, atk_preds, labels_all = [], [], []

    for ids, masks, labels in tqdm(loader, desc="Robust", leave=False):
        B, N, L = ids.shape
        # use only the original sentence (index 0) for clean eval
        clean_ids  = ids[:, 0:1, :]
        clean_mask = masks[:, 0:1, :]

        clean_out = model(clean_ids.to(device), clean_mask.to(device))
        clean_p   = clean_out["logits"].argmax(-1).cpu().tolist()

        # generate SCPN-like attacked variants and re-encode
        # (we must detokenize; for speed we use the raw text from augment)
        # In a real pipeline you'd store original texts in the batch
        # Here we re-augment from the decoded token ids as a proxy
        atk_ids_list, atk_mask_list = [], []
        for b in range(B):
            # decode original sentence
            tok_ids = ids[b, 0].tolist()
            text = tokenizer.decode(tok_ids, skip_special_tokens=True)
            atk_text = simulate_scpn_attack(text)
            enc = tokenizer(
                atk_text, truncation=True, max_length=max_len,
                padding="max_length", return_tensors="pt"
            )
            atk_ids_list.append(enc["input_ids"])
            atk_mask_list.append(enc["attention_mask"])

        atk_ids  = torch.cat(atk_ids_list, dim=0).unsqueeze(1).to(device)
        atk_mask = torch.cat(atk_mask_list, dim=0).unsqueeze(1).to(device)

        atk_out  = model(atk_ids, atk_mask)
        atk_p    = atk_out["logits"].argmax(-1).cpu().tolist()

        clean_preds.extend(clean_p)
        atk_preds.extend(atk_p)
        labels_all.extend(labels.tolist())

    clean_acc  = accuracy_score(labels_all, clean_preds)
    atk_acc    = accuracy_score(labels_all, atk_preds)
    flip_rate  = sum(c != a for c, a in zip(clean_preds, atk_preds)) / len(clean_preds)

    return {
        "clean_acc":  clean_acc,
        "attacked_acc": atk_acc,
        "flip_rate":  flip_rate,
    }


# ──────────────────────────────────────────────────────────────────────────────
# 7.  PSEUDOCODE DOCUMENTATION (printed at runtime)
# ──────────────────────────────────────────────────────────────────────────────

PSEUDOCODE = """
╔══════════════════════════════════════════════════════════════════════════════╗
║              SEIR-DNet TRAINING PSEUDOCODE                                  ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  INPUT : sentence s, label y                                                 ║
║  HYPERPARAMS: N=3, λ₁=0.5, λ₂=0.3, λ₃=0.1, T=2.0                          ║
║                                                                              ║
║  ── AUGMENTATION ──────────────────────────────────────────────────────────  ║
║  variants = [s, paraphrase(s), back_translate(s)]    # N = 3 views          ║
║                                                                              ║
║  ── ENCODING (shared BERT + HRL) ───────────────────────────────────────── ║
║  FOR i IN 0..N-1:                                                            ║
║    cls_i, hidden_i = BERT(variants[i])                                       ║
║    z_i = HRL(hidden_i, cls_i)     # word+phrase+sentence fusion              ║
║                                                                              ║
║  ── SAM: CROSS-VARIANT ALIGNMENT ───────────────────────────────────────── ║
║  Z = stack([z_0, z_1, ..., z_{N-1}])          # (B, N, D)                   ║
║  Z_aligned = SAM(Z)                            # cross-attention             ║
║    A = softmax(Q·Kᵀ / √D) · V                                               ║
║    Z_aligned = LayerNorm(Z + A)                                              ║
║                                                                              ║
║  ── LSCR: LATENT SEMANTIC CONSISTENCY ─────────────────────────────────── ║
║  ẑ_i = W_proj(Z_aligned_i)                     # project to lower dim       ║
║  L_sim    = (1/N) Σᵢ [ MSE(ẑ₀, ẑᵢ) + (1 - cos(ẑ₀, ẑᵢ)) ]                 ║
║  L_struct = Var(‖ẑᵢ‖₂ over i)                  # norm consistency           ║
║                                                                              ║
║  ── CLASSIFICATION + PVN ───────────────────────────────────────────────── ║
║  FOR i IN 0..N-1:                                                            ║
║    logits_i = Classifier(Z_aligned_i)                                        ║
║    p_i      = softmax(logits_i / T)                                          ║
║  L_consist  = (1/N) Σᵢ [KL(p₀‖pᵢ) + KL(pᵢ‖p₀)] / 2                        ║
║                                                                              ║
║  ── COMBINED LOSS ──────────────────────────────────────────────────────── ║
║  L_cls   = CrossEntropy(logits_0, y)                                         ║
║  L_total = L_cls + λ₁·L_consist + λ₂·L_sim + λ₃·L_struct                   ║
║                                                                              ║
║  ── BACKPROPAGATION ────────────────────────────────────────────────────── ║
║  L_total.backward()                                                          ║
║  optimizer.step()                                                            ║
║                                                                              ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  INFERENCE (defense):                                                        ║
║    Given adversarial input s_adv:                                            ║
║    z_adv = HRL(BERT(s_adv))       # single forward pass                      ║
║    logits = Classifier(SAM(z_adv.unsqueeze(1))[:, 0, :])                     ║
║    ↑  LSCR-trained encoder maps s_adv to same region as s_orig               ║
║       even if syntax/lexicon differ → correct prediction                     ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

MATH_FORMULATION = """
╔══════════════════════════════════════════════════════════════════════════════╗
║                    MATHEMATICAL FORMULATION                                 ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  Notation:                                                                   ║
║    s₀            : original sentence                                         ║
║    {s₁,...,s_{N-1}}: augmented variants (paraphrases, transforms)            ║
║    z_i = f_θ(sᵢ) : HRL(BERT(sᵢ)) — sentence embedding                       ║
║    ẑ_i = W·z_i   : LSCR projection                                           ║
║    p_i           : softmax(classifier(SAM(z_i)) / T)                         ║
║                                                                              ║
║  SAM (cross-variant attention):                                              ║
║    Q = W_q · Z,  K = W_k · Z,  V = W_v · Z                                  ║
║    A = softmax(Q·Kᵀ / √d_k)                                                  ║
║    Z' = LayerNorm(Z + A·V)       [residual cross-attention]                  ║
║                                                                              ║
║  HRL (hierarchical fusion):                                                  ║
║    word  = MaskedMeanPool(BERT_hidden)                                       ║
║    phrase= MaskedMeanPool(Conv1D(BERT_hidden, k=3))                          ║
║    sent  = CLS_embedding                                                     ║
║    z_i   = MLP([word; phrase; sent])                                         ║
║                                                                              ║
║  LSCR losses:                                                                ║
║    L_sim    = (1/N-1) Σᵢ₌₁ᴺ [ ‖ẑ₀ - ẑᵢ‖² + (1 - cos(ẑ₀,ẑᵢ)) ]            ║
║    L_struct = Var_i(‖ẑᵢ‖₂)                                                  ║
║                                                                              ║
║  PVN (prediction consistency):                                               ║
║    L_consist = (1/N-1) Σᵢ₌₁ᴺ [KL(p₀‖pᵢ) + KL(pᵢ‖p₀)] / 2                  ║
║                                                                              ║
║  Total loss:                                                                 ║
║    L_total = L_cls + λ₁·L_consist + λ₂·L_sim + λ₃·L_struct                  ║
║    default: λ₁=0.5, λ₂=0.3, λ₃=0.1                                          ║
║                                                                              ║
║  Convergence condition:                                                      ║
║    d(z_i, z_j) → 0  ∀ semantically equivalent (sᵢ, sⱼ)                      ║
║    KL(p_i, p_j) → 0  ∀ (sᵢ, sⱼ) in same equivalence class                  ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

COMPARISON_TABLE = """
╔════════════════════╦════════════════════╦═════════════════════╦══════════════╗
║ Method             ║ Defense level      ║ Handles SCPN/SEA?   ║ Extra cost   ║
╠════════════════════╬════════════════════╬═════════════════════╬══════════════╣
║ BERT (baseline)    ║ None               ║ No                  ║ —            ║
║ Adversarial train  ║ Token-level        ║ Partial             ║ 2×           ║
║ R-Drop             ║ Stochastic dropout ║ No (same tokens)    ║ 2×           ║
║ Contrastive (SimCSE)║ Sentence-level    ║ Partial             ║ 2×           ║
║ SEIR-DNet (ours)   ║ Sentence-level +  ║ YES (by design)     ║ N× (N=3)     ║
║                    ║ structural         ║                     ║              ║
╚════════════════════╩════════════════════╩═════════════════════╩══════════════╝
Key differences vs. contrastive learning:
  • LSCR is a deterministic architectural module, not a loss on negative pairs
  • SAM explicitly re-encodes each variant conditioned on all others
  • PVN enforces prediction-level consistency, not just embedding-level
  • HRL captures multi-granularity structure (word/phrase/sentence)
"""


# ──────────────────────────────────────────────────────────────────────────────
# 8.  MAIN — TRAINING + EVALUATION
# ──────────────────────────────────────────────────────────────────────────────

def main():
    # ── CONFIG ──────────────────────────────────────────────────────────────
    CFG = dict(
        bert_name       = "bert-base-uncased",
        max_len         = 64,        # 64 tokens covers most SST-2 sentences; 128 for full runs
        n_variants      = 3,
        batch_size      = 16,        # reduce to 8 if GPU OOM
        epochs          = 2,         # 2 epochs is enough to validate the model works
        lr              = 2e-5,
        warmup_ratio    = 0.1,
        weight_decay    = 0.01,
        lambda_consist  = 0.5,
        lambda_sim      = 0.3,
        lambda_struct   = 0.1,
        pvn_temp        = 2.0,
        proj_dim        = 256,
        sam_heads       = 4,
        dropout         = 0.1,
        num_labels      = 2,
        eval_robustness = True,
        save_path       = "seir_dnet_best.pt",
        # ── QUICK DEMO: 300 train / 60 val ──────────────────────────────────
        # Set quick_demo=False and epochs=3+ for a proper training run.
        quick_demo      = True,
        train_size      = 300,
        val_size        = 60,
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{'='*70}")
    print("  SEIR-DNet: Sentence-level Adversarial Defense")
    print(f"  Device: {device}")
    print(f"{'='*70}\n")

    # Print documentation
    print(PSEUDOCODE)
    print(MATH_FORMULATION)
    print(COMPARISON_TABLE)

    # ── TOKENIZER + DATASETS ────────────────────────────────────────────────
    print("[1/4] Loading tokenizer and datasets...")
    tokenizer = BertTokenizer.from_pretrained(CFG["bert_name"])

    train_ds = SST2Dataset("train",      tokenizer, CFG["max_len"], CFG["n_variants"])
    val_ds   = SST2Dataset("validation", tokenizer, CFG["max_len"], CFG["n_variants"])

    if CFG["quick_demo"]:
        train_ds.texts  = train_ds.texts[:CFG["train_size"]]
        train_ds.labels = train_ds.labels[:CFG["train_size"]]
        val_ds.texts    = val_ds.texts[:CFG["val_size"]]
        val_ds.labels   = val_ds.labels[:CFG["val_size"]]

    train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"],
                              shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"],
                              shuffle=False, num_workers=2, pin_memory=True)

    print(f"    Train: {len(train_ds):,} samples | Val: {len(val_ds):,} samples")

    # ── MODEL + OPTIMIZER + SCHEDULER ───────────────────────────────────────
    print("[2/4] Building SEIR-DNet model...")
    model = SEIRDNet(
        bert_name   = CFG["bert_name"],
        num_labels  = CFG["num_labels"],
        n_variants  = CFG["n_variants"],
        proj_dim    = CFG["proj_dim"],
        sam_heads   = CFG["sam_heads"],
        pvn_temp    = CFG["pvn_temp"],
        dropout     = CFG["dropout"],
    ).to(device)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"    Trainable parameters: {n_params:,}")

    criterion = SEIRLoss(
        lambda_consist = CFG["lambda_consist"],
        lambda_sim     = CFG["lambda_sim"],
        lambda_struct  = CFG["lambda_struct"],
    )

    # Separate learning rates: BERT layers get lower LR (fine-tuning)
    bert_params  = list(model.bert.parameters())
    other_params = [p for n, p in model.named_parameters()
                    if not n.startswith("bert.")]
    optimizer = torch.optim.AdamW(
        [
            {"params": bert_params,  "lr": CFG["lr"]},
            {"params": other_params, "lr": CFG["lr"] * 5},   # novel components 5× LR
        ],
        weight_decay=CFG["weight_decay"],
    )

    total_steps   = len(train_loader) * CFG["epochs"]
    warmup_steps  = int(total_steps * CFG["warmup_ratio"])
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    # ── TRAINING ─────────────────────────────────────────────────────────────
    print(f"\n[3/4] Training for {CFG['epochs']} epochs...\n")
    best_val_acc = 0.0
    history = []

    for epoch in range(1, CFG["epochs"] + 1):
        t0 = time.time()
        train_m = train_epoch(model, train_loader, optimizer, scheduler,
                              criterion, device)
        val_m   = evaluate(model, val_loader, criterion, device)
        elapsed = time.time() - t0

        history.append({"epoch": epoch, "train": train_m, "val": val_m})

        print(
            f"  Epoch {epoch:02d}/{CFG['epochs']:02d} | "
            f"Train loss={train_m['loss']:.4f} acc={train_m['acc']:.4f} | "
            f"Val loss={val_m['loss']:.4f} acc={val_m['acc']:.4f} | "
            f"Time={elapsed:.1f}s"
        )
        print(
            f"           Loss breakdown — "
            f"cls={train_m['cls']:.4f}  "
            f"consist={train_m['consist']:.4f}  "
            f"sim={train_m['sim']:.4f}  "
            f"struct={train_m['struct']:.4f}"
        )

        if val_m["acc"] > best_val_acc:
            best_val_acc = val_m["acc"]
            torch.save(model.state_dict(), CFG["save_path"])
            print(f"           ✓ New best model saved ({best_val_acc:.4f})")

    # ── ROBUSTNESS EVALUATION ────────────────────────────────────────────────
    if CFG["eval_robustness"]:
        print(f"\n[4/4] Robustness evaluation (SCPN-simulated attacks)...")
        model.load_state_dict(torch.load(CFG["save_path"], map_location=device))
        rob = evaluate_robustness(model, val_loader, tokenizer,
                                  criterion, device, CFG["max_len"])
        print(f"\n  ┌─────────────────────────────────────────┐")
        print(f"  │  Clean accuracy    : {rob['clean_acc']:.4f}             │")
        print(f"  │  Attacked accuracy : {rob['attacked_acc']:.4f}             │")
        print(f"  │  Prediction flip   : {rob['flip_rate']:.4f}             │")
        print(f"  └─────────────────────────────────────────┘")

    print("\n" + "="*70)
    print("  Training complete!")
    print(f"  Best validation accuracy : {best_val_acc:.4f}")
    print("="*70)

    # ── ADVANTAGES / LIMITATIONS SUMMARY ────────────────────────────────────
    print("""
EXPECTED ADVANTAGES of SEIR-DNet:
  ✓ Handles full-sentence paraphrase attacks (SCPN, SEA, TextFooler-sentence)
  ✓ No need for explicit attack generation at training time (unlike AT)
  ✓ SAM + LSCR together enforce both embedding AND prediction consistency
  ✓ HRL captures sub-sentence structure → richer representations
  ✓ PVN is computationally cheap (KL on softmax outputs only)
  ✓ Plug-and-play on top of any pretrained BERT variant

LIMITATIONS:
  ✗ N×batch size increases memory usage (mitigate: gradient checkpointing)
  ✗ Augmentation quality affects robustness (toy augmentation < real SCPN)
  ✗ λ hyperparameters require tuning per dataset
  ✗ Very short sentences (<5 tokens) benefit less from HRL phrase-level
  ✗ Does not handle image/multimodal adversarial attacks
""")


if __name__ == "__main__":
    main()


  SEIR-DNet: Sentence-level Adversarial Defense
  Device: cuda


╔══════════════════════════════════════════════════════════════════════════════╗
║              SEIR-DNet TRAINING PSEUDOCODE                                  ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  INPUT : sentence s, label y                                                 ║
║  HYPERPARAMS: N=3, λ₁=0.5, λ₂=0.3, λ₃=0.1, T=2.0                          ║
║                                                                              ║
║  ── AUGMENTATION ──────────────────────────────────────────────────────────  ║
║  variants = [s, paraphrase(s), back_translate(s)]    # N = 3 views          ║
║                                                                              ║
║  ── ENCODING (shared BERT + HRL) ───────────────────────────────────────── ║
║  FOR i IN 0..N-1:                               

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

    Trainable parameters: 119,490,178

[3/4] Training for 2 epochs...



  Epoch 01/02 | Train loss=0.6536 acc=0.5967 | Val loss=0.3721 acc=0.9000 | Time=12.0s
           Loss breakdown — cls=0.6319  consist=0.0021  sim=0.0691  struct=0.0000
           ✓ New best model saved (0.9000)


  Epoch 02/02 | Train loss=0.3026 acc=0.9033 | Val loss=0.2190 acc=0.9167 | Time=10.5s
           Loss breakdown — cls=0.2846  consist=0.0190  sim=0.0285  struct=0.0000
           ✓ New best model saved (0.9167)

[4/4] Robustness evaluation (SCPN-simulated attacks)...


Robust:   0%|          | 0/4 [00:00<?, ?it/s]/tmp/ipykernel_6130/1086705201.py:359: UserWarning: var(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  l_struct = all_norms.var(dim=1).mean()
                                                     


  ┌─────────────────────────────────────────┐
  │  Clean accuracy    : 0.9000             │
  │  Attacked accuracy : 0.8167             │
  │  Prediction flip   : 0.1500             │
  └─────────────────────────────────────────┘

  Training complete!
  Best validation accuracy : 0.9167

EXPECTED ADVANTAGES of SEIR-DNet:
  ✓ Handles full-sentence paraphrase attacks (SCPN, SEA, TextFooler-sentence)
  ✓ No need for explicit attack generation at training time (unlike AT)
  ✓ SAM + LSCR together enforce both embedding AND prediction consistency
  ✓ HRL captures sub-sentence structure → richer representations
  ✓ PVN is computationally cheap (KL on softmax outputs only)
  ✓ Plug-and-play on top of any pretrained BERT variant

LIMITATIONS:
  ✗ N×batch size increases memory usage (mitigate: gradient checkpointing)
  ✗ Augmentation quality affects robustness (toy augmentation < real SCPN)
  ✗ λ hyperparameters require tuning per dataset
  ✗ Very short sentences (<5 tokens) benefit less from H